In [2]:
# Scenario: AI System for Detecting Fraud in Financial Transactions
# 🚨 The Problem
# A major bank processes millions of transactions daily.
# Fraud analysts are:
# - Overwhelmed by the sheer volume
# - Limited in number
# - Required to make instant decisions
# - Missing subtle fraud patterns hidden in massive datasets
# Critical cases like:
# 👉 Credit card fraud
# 👉 Money laundering
# 👉 Normal transactions
# must be flagged in real-time.
# Delays can cost millions and damage customer trust.

# 💡 The Solution: Build an AI Assistant
# The bank deploys an AI model that can pre-screen transactions and alert analysts.
# But here’s the challenge…
# ❗ Fraud datasets are usually imbalanced and small.
# Training a deep neural network from scratch would require:
# - Billions of labeled transactions
# - Huge compute clusters
# - Months of training
# Not practical.

# ⭐ Enter Transfer Learning (Your Code)
# Instead of starting from zero, engineers use:
# 👉 BERT (Bidirectional Encoder Representations from Transformers) trained on massive text corpora.
# Although BERT was trained on general language (Wikipedia, books, etc.), its early layers learn universal text patterns, like:
# - ✅ Word embeddings
# - ✅ Sentence structures
# - ✅ Semantic relationships
# - ✅ Contextual meaning
# These features are also present in transaction descriptions, merchant details, and customer behavior logs.
# By fine-tuning BERT on the bank’s fraud dataset, the AI can quickly learn to distinguish:
# - Suspicious vs. normal transactions
# - Fraud rings vs. genuine customer activity

# ⚡ Just like ResNet50 helps radiologists with X-rays, BERT helps fraud analysts with transaction logs —
# both leveraging transfer learning to solve problems where data is scarce but decisions are critical.

In [3]:
pip install transformers torch pandas scikit-learn datasets

In [4]:
import pandas as pd

data = {
    "text": [
        "Payment at Amazon store",
        "Transfer to offshore account Cayman Islands",
        "Purchase at Walmart",
        "Multiple rapid withdrawals from ATM",
        "Normal grocery store purchase"
    ],
    "label": [0, 2, 0, 1, 0]  # 0=normal, 1=fraud, 2=money laundering
}

df = pd.DataFrame(data)
print(df)

                                          text  label
0                      Payment at Amazon store      0
1  Transfer to offshore account Cayman Islands      2
2                          Purchase at Walmart      0
3          Multiple rapid withdrawals from ATM      1
4                Normal grocery store purchase      0


In [5]:
from transformers import BertTokenizer, BertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

dataset = dataset.map(tokenize)
dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [7]:
import torch

def predict_transaction(text):
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()

    labels = {
        0: "Normal Transaction",
        1: "Fraud",
        2: "Money Laundering"
    }

    return labels[prediction]

In [8]:
transaction = "Transfer of $9000 to offshore account"

result = predict_transaction(transaction)

print("Transaction:", transaction)
print("Prediction:", result)

Transaction: Transfer of $9000 to offshore account
Prediction: Fraud
